# The Registry

**Prerequisites:** `01_what_is_a_tool.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

Notebook 1 ended with three dicts, built by hand from one list of tools. Each part of chapter 2's loop needs a different one of them.

With two tools that is fine. This notebook is about what happens when there are thirty. Two of them turn out to have the same name. One has a name the API won't take. And all thirty are described to the model on every single turn, whether they get used or not.

**By the end you'll have:**
- Built `Registry`, which owns those three views instead of you rebuilding them
- Watched a plain dict quietly throw away one of two tools with the same name, and made the registry refuse instead
- Given two tools with the same name a prefix so both can be used
- Tested what the API actually accepts as a tool name, and found it stricter in one place and looser in another than you'd expect
- Measured what a thirty-tool toolbox costs in tokens, on every turn
- Tried twice to catch a thirty-tool toolbox spoiling the model's choices, failed both times, and worked out what that does and doesn't settle
- Cut a registry down by name and by tag, and run the result in chapter 2's ReAct loop

## Setup

In [1]:
import json
import sys
from pathlib import Path
from typing import Annotated

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))

from chat import count_tokens
from llm import client, DEFAULT_MODEL
from registry import Registry
from tools import Tool, tool

## Where notebook 1 left off

Two tools, and the three views of them the loop needs:

In [2]:
@tool
def get_weather(city: Annotated[str, "The city to look up"]) -> str:
    """Look up the current weather in a city."""
    return f"4 degrees celsius, clear skies in {city}."

@tool
def search_wiki(query: Annotated[str, "What to search for"]) -> str:
    """Search the company wiki."""
    return f"wiki results for: {query}"

TOOLS = [get_weather, search_wiki]

schemas_for_model = [t.schema for t in TOOLS]    # what Think sends the API
functions = {t.name: t.function for t in TOOLS}  # what Act calls
schemas = {t.name: t.args for t in TOOLS}        # what Act checks against

print(list(functions))

['get_weather', 'search_wiki']


Three lines is not much work. The problem is that they are three lines you have to remember to keep in step, and they only exist because each part of the loop wants the tools in a different shape.

A registry owns all three. You add a tool once and ask for whichever view you need:

```python
class Registry:
    @property
    def tools(self):
        return [tool.schema for tool in self._tools.values()]

    @property
    def functions(self):
        return {name: tool.function for name, tool in self._tools.items()}

    @property
    def schemas(self):
        return {name: tool.args for name, tool in self._tools.items()}
```

**One thing about those names is worth saying out loud, because it reads backwards.** `tools` here means the JSON the model reads. `schemas` means the Pydantic models that check arguments. That is not the obvious way round.

It is deliberate. Those are the names chapter 2 already uses — `Think(tools=...)` and `Act(schemas=...)` — and they in turn come from the API. Renaming them here would read better in this one file and worse at every place it gets wired up. The actual `Tool` objects are reached by name instead, with `registry["get_weather"]`, or by looping over the registry.

In [3]:
registry = Registry([get_weather, search_wiki])

print(registry)
print("one tool:", registry["get_weather"])
print("is it there:", "search_wiki" in registry, "| how many:", len(registry))
print("names in the JSON:", [t["function"]["name"] for t in registry.tools])
print("argument checks:", {name: model.__name__ for name, model in registry.schemas.items()})

<Registry 2 tools: get_weather, search_wiki>
one tool: <Tool get_weather(city)>
is it there: True | how many: 2
names in the JSON: ['get_weather', 'search_wiki']
argument checks: {'get_weather': 'get_weather_args', 'search_wiki': 'search_wiki_args'}


A plain function can be added too. It gets wrapped in a `Tool` on the way in, so `@tool` doesn't have to be there at the point the function was written. That matters for functions coming from a library you don't control:

In [4]:
def check_status(service: Annotated[str, "Name of the service"]) -> str:
    """Check whether a service is up."""
    return f"{service} is up."

registry.add(check_status)
print(registry.names)

['get_weather', 'search_wiki', 'check_status']


## Two tools with the same name

`search` is the obvious example. A documentation search and a ticket search are different tools, both want to be called `search`, and they usually arrive from different places — which means nobody is looking at them side by side.

Here is what a plain dict does about that:

In [5]:
def search_docs(query: str) -> str:
    return f"doc results for: {query}"

def search_tickets(query: str) -> str:
    return f"ticket results for: {query}"

by_name = {}
by_name["search"] = search_docs
by_name["search"] = search_tickets     # same key, no complaint

print("two tools registered, one left:", by_name["search"](query="refund policy"))

two tools registered, one left: ticket results for: refund policy


Nothing went wrong, as far as Python is concerned. Both tools were registered. One of them is gone.

This is worse than a crash. The model asks for `search`, gets an answer, and the answer is confident and about the wrong thing — ticket results when it wanted documentation. Nothing in the conversation looks like an error, so nothing retries, and the run continues on a wrong footing.

`Registry.add` refuses instead:

In [6]:
docs = Registry()
docs.add(search_docs, name="search")

tickets = Registry()
tickets.add(search_tickets, name="search")

try:
    docs.merge(tickets)
except ValueError as e:
    print(e)

a tool named 'search' is already registered. Pass prefix= or name= to give this one a different name.


## Prefixes

Refusing is only useful if there is something to do about it. `prefix=` renames a tool as it goes in, so both can live in the same registry:

In [7]:
combined = Registry()
combined.merge(docs, prefix="docs")
combined.merge(tickets, prefix="tickets")

print(combined.names)
print("what the model is shown:", [t["function"]["name"] for t in combined.tools])
print("the tool it was made from is untouched:", tickets["search"].name)

['docs_search', 'tickets_search']
what the model is shown: ['docs_search', 'tickets_search']
the tool it was made from is untouched: search


The rename produces a copy. The name is part of what the model reads — `Tool.schema` is built from it — and it has to match the key the call comes back under. Renaming the original object instead would change it for everyone else holding a reference to the same tool.

## What counts as a name

Prefixing raises a question worth answering with a real request rather than a guess: what is the API willing to accept as a tool name? A dot would read nicely (`docs.search`), so it is worth finding out whether it is allowed.

In [8]:
def name_schema(name):
    return {"type": "function", "function": {
        "name": name,
        "description": "Search the internal documentation.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]},
    }}

for candidate in ["docs.search", "docs search", "docs_search"]:
    try:
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=[{"role": "user", "content": "Find the docs on pricing."}],
            tools=[name_schema(candidate)],
            max_tokens=60,
        )
        called = response.choices[0].message.tool_calls
        print(f"{candidate!r}: accepted, model called {called[0].function.name!r}")
    except Exception as e:
        print(f"{candidate!r}: rejected - {str(e)[:160]}")

'docs.search': accepted, model called 'docs.search'
'docs search': rejected - Error code: 400 - {'error': {'message': 'code=400, message=tool `docs search` cannot contain whitespace in its name, type=invalid_request_error', 'type': 'inval


'docs_search': accepted, model called 'docs_search'


Two things there, and only one of them is what you'd expect.

**Whitespace is rejected, and it takes the whole request with it.** Not that one tool — the request. Every other tool in it fails too, and the error says nothing about which agent or which turn produced it. That is the argument for checking names when a tool is registered rather than when a request is sent.

**The dot is accepted here.** OpenAI's documented rule for tool names is letters, digits, underscores and hyphens, up to 64 characters. Groq is more relaxed than that in practice.

`Registry` follows the stricter rule anyway:

```python
NAME = re.compile(r"^[A-Za-z0-9_-]{1,64}$")
```

That is a choice, not a fact about the API, so it is worth defending. A tool defined in this repo should keep working when it is pointed at a different provider, and the strictest common rule is the only one that survives the move. Being stricter than the provider you happen to be on today costs one underscore.

In [9]:
for bad in ["docs.search", "get weather", "search/web"]:
    try:
        Registry().add(get_weather, name=bad)
    except ValueError as e:
        print(e)

'docs.search' cannot be used as a tool name: only letters, digits, underscores and hyphens are allowed, up to 64 characters.
'get weather' cannot be used as a tool name: only letters, digits, underscores and hyphens are allowed, up to 64 characters.
'search/web' cannot be used as a tool name: only letters, digits, underscores and hyphens are allowed, up to 64 characters.


## What a big toolbox costs

Everything so far shows up with three tools. This part only shows up with thirty.

Here is a plausible company toolbox. The three email tools are written out properly, because they are what the next section tests. The rest are stubs — the bodies do nothing, and what matters is only how they look to the model:

In [10]:
@tool
def draft_email(to: Annotated[str, "Who the email is addressed to"],
                body: Annotated[str, "The text of the email"]) -> str:
    """Write an email and save it as a draft, without sending it."""
    return f"Draft saved, addressed to {to}."

@tool
def send_email(to: Annotated[str, "Who to send the email to"],
               body: Annotated[str, "The text of the email"]) -> str:
    """Send an email immediately."""
    return f"Email sent to {to}."

@tool
def schedule_email(to: Annotated[str, "Who to send the email to"],
                   body: Annotated[str, "The text of the email"],
                   send_at: Annotated[str, "When to send it, e.g. '2026-08-20 09:00'"]) -> str:
    """Send an email later, at a given time."""
    return f"Email to {to} scheduled for {send_at}."

PADDING = [
    ("search_docs", "Search the internal documentation."),
    ("search_tickets", "Search the issue tracker for bug reports and feature requests."),
    ("search_crm", "Search customer records by company or contact name."),
    ("search_code", "Search the source code repositories."),
    ("search_chat", "Search the team chat history."),
    ("list_calendar", "List the events on someone's calendar for a given day."),
    ("book_meeting", "Book a meeting room and invite attendees."),
    ("cancel_meeting", "Cancel a meeting that has already been booked."),
    ("create_ticket", "Open a new ticket in the issue tracker."),
    ("close_ticket", "Close a ticket in the issue tracker."),
    ("assign_ticket", "Assign a ticket to a team member."),
    ("get_invoice", "Look up an invoice by number."),
    ("issue_refund", "Refund a customer's payment."),
    ("get_account", "Look up a customer account by name."),
    ("update_account", "Change the details on a customer account."),
    ("list_deploys", "List recent deployments to production."),
    ("start_deploy", "Deploy the current build to production."),
    ("rollback_deploy", "Roll production back to the previous build."),
    ("get_metrics", "Fetch service metrics for a time range."),
    ("get_logs", "Fetch service logs for a time range."),
    ("page_oncall", "Page whoever is on call right now."),
    ("create_doc", "Create a new document in the shared drive."),
    ("share_doc", "Share a document with someone."),
    ("export_report", "Export a report as a spreadsheet."),
    ("list_employees", "List employees in a team."),
    ("book_leave", "Book annual leave for an employee."),
    ("submit_expense", "Submit an expense claim."),
]

toolbox = Registry([draft_email, send_email, schedule_email])
for name, description in PADDING:
    toolbox.add(Tool(lambda query: query, name=name, description=description,
                     args=search_wiki.args), tags=["office"])

small = Registry([get_weather, search_wiki])

print(f"{len(small)} tools: {count_tokens(json.dumps(small.tools)):>5} tokens")
print(f"{len(toolbox)} tools: {count_tokens(json.dumps(toolbox.tools)):>5} tokens")

2 tools:   145 tokens
30 tools:  2274 tokens


That is the cost of *having* the tools, before anything is asked or answered.

And it is paid per turn, not per run. Chapter 2's ReAct loop sends the whole tool list again on every pass, so a five-step run pays it five times. Chapter 1's notebook 3 measured the same thing from the other direction: tokens are the budget, and this is a fixed charge on every request an agent makes for the rest of its run.

## Can it still pick the right one?

Tokens are the measurable cost. The other one is harder to pin down: the more tools there are, the more chances there are to pick a near-miss.

So here is a question with a right answer and two tempting wrong ones. "Write an email, but don't send it yet" should be `draft_email`. `send_email` and `schedule_email` are both sitting right next to it:

In [11]:
question = "Write an email to Priya about the Friday deploy, but don't send it yet."

def ask(registry, prompt=question):
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        tools=registry.tools,
    )
    calls = response.choices[0].message.tool_calls
    return calls[0].function.name if calls else "(answered without calling anything)"

print(f"with all {len(toolbox)} tools:", ask(toolbox))

with all 30 tools: draft_email


It picked the right one. "Don't send it yet" calls for `draft_email`, and having twenty-nine other tools sitting next to it made no difference.

That is worth stating plainly, because the other outcome would have made a better story: **the model was not confused by the extra tools.** One question, one model, thirty tools, chosen correctly.

The other way a big toolbox goes wrong is the model reaching for a tool when it didn't need one at all. Chapter 1's notebook 6 saw a version of this: offered `add` and `multiply` and asked an unrelated question, the model tried to call a search tool it had never been given.

Thirty tools is thirty chances for something to look relevant. Here is a question none of them can help with:

In [12]:
trivia = "What is the capital of France?"

print("with 2 tools: ", ask(small, trivia))
print("with 30 tools:", ask(toolbox, trivia))

with 2 tools:  (answered without calling anything)


with 30 tools: (answered without calling anything)


Nothing went wrong here either. Neither registry tempted it into a call, and both answered in plain text.

So: two attempts at showing that a big toolbox spoils the model's choices, and neither one showed it. That is the honest result, and it decides what the rest of this notebook is allowed to claim.

What is certain is the token cost, because it was counted rather than sampled — 2,274 tokens against 145, on every turn, before the model does anything with them.

The accuracy worry isn't disproved either. Thirty tools is not many, `openai/gpt-oss-120b` is a capable model, and two questions is not a measurement. It is simply not something these cells established, so this notebook doesn't lean on it. Subsets are worth building for the cost that *was* measured.

## Subsets

Whichever way those went, the decision is the same: an agent should be handed the tools it needs, not everything that exists. That is what `subset` is for. Tools can be picked out by name, or by a tag given when they were registered:

In [13]:
email_tools = toolbox.subset(["draft_email", "send_email", "schedule_email", "list_calendar"])
print("by name:", email_tools.names)
print("by tag: ", len(toolbox.subset(tags=["office"])), "tools tagged office")

# a name that isn't registered is a mistake, not an empty result
try:
    toolbox.subset(["draft_emails"])
except KeyError as e:
    print("typo:", e)

print()
print(f"{len(toolbox)} tools: {count_tokens(json.dumps(toolbox.tools)):>5} tokens")
print(f"{len(email_tools)} tools: {count_tokens(json.dumps(email_tools.tools)):>5} tokens")
print()
print(f"with {len(email_tools)} tools:", ask(email_tools))

by name: ['draft_email', 'send_email', 'schedule_email', 'list_calendar']
by tag:  27 tools tagged office
typo: "no tool named 'draft_emails' is registered"

30 tools:  2274 tokens
4 tools:   408 tokens



with 4 tools: draft_email


A typo in a name raises instead of quietly returning a smaller registry. The quiet version produces an agent that is simply missing a tool, and the run then goes wrong later, somewhere else, for reasons that look nothing like a misspelled name.

## Into the loop

`Registry` gives each part of chapter 2's loop the view it wants, so the wiring is now the registry three times, and `02_agent_runtime` still doesn't change:

In [14]:
from chat import Conversation
from runtime import Runtime
from observe import Observe
from think import Think
from act import Act

@tool
def sunset_time(city: Annotated[str, "The city to look up"]) -> str:
    """The local time the sun sets today in a city."""
    at = {"tokyo": "18:12", "oslo": "21:47", "cairo": "17:55"}.get(city.lower())
    return f"The sun sets at {at} in {city}." if at else f"No sunset data for {city}."

outdoors = Registry([get_weather, sunset_time])

runtime = Runtime(
    loop=[
        Observe(event="What's the weather in Oslo, and when does the sun set there?"),
        Think(tools=outdoors.tools, allow_fork=False),
        Act(functions=outdoors.functions, schemas=outdoors.schemas),
        Observe(),
    ],
    repeat_from=1,
)

conversation = runtime.run(Conversation())
for m in conversation.messages:
    print(f"- {m['role']:9} {m['content']!r}")

- user      "What's the weather in Oslo, and when does the sun set there?"
- decision  '{"city":"Oslo"}'
- tool      'The temperature in Oslo is 4\u202f°C with clear skies.'
- decision  '{"city":"Oslo"}'
- tool      'The sun sets at 21:47 in Oslo.'
- decision  'The weather in Oslo is 4\u202f°C with clear skies, and the sun sets at 21:47.'
- assistant 'The weather in Oslo is 4\u202f°C with clear skies, and the sun sets at 21:47.'


What the registry still does not do is decide *which* subset an agent gets. Every subset in this notebook was chosen by hand, by someone who already knew what the agent was going to be asked.

Choosing tools from the request itself — thirty registered, four sent — is a retrieval problem, and it needs the machinery `05_retrieval` builds. The registry is what makes it possible later: the tools are in one place, with names and descriptions and tags, instead of spread across three dicts.

## What you built

✓ Built `Registry`: add a tool once, and get whichever of the three views the runtime asks for

✓ Kept chapter 2's names (`tools` for the JSON, `schemas` for the argument checks) even though they read backwards, so the wiring stays readable at the call site

✓ Watched a plain dict take two tools called `search` and keep one, with no error — a wrong answer delivered confidently, which nothing in the loop can notice

✓ Made registration refuse a name that is already taken, and added `prefix=` so both tools can be kept

✓ Found out what the API really accepts as a tool name: whitespace is rejected and takes the entire request with it, while a dot is accepted here even though OpenAI's rule doesn't allow it — and chose the stricter rule on purpose, so a tool still works on another provider

✓ Measured a thirty-tool toolbox against a two-tool one, and remembered that the cost is charged on every turn of every run

✓ Tried twice to catch a thirty-tool toolbox spoiling the model's choices — a pick between `draft_email`, `send_email` and `schedule_email`, and a question no tool could answer — and saw it get both right. The case for subsets rests on the token count, which was measured, not on a failure that didn't happen

✓ Cut a registry down by name and by tag, with a typo raising instead of silently returning less

✓ Wired the whole thing into chapter 2's ReAct loop, still without changing `02_agent_runtime`

**Next:** `03_invocation.ipynb` — what happens at the moment a tool actually runs. A call that never returns, a failure that should be handed back to the model instead of raised, a result far too big to put in a conversation, and a call that is not safe to retry.